In [ ]:
pip install requests

In [ ]:
pip install pandas

In [ ]:
import requests
import pandas as pd

In [ ]:
records = []

for page in range (1,7):

  url = (f"https://api.coingecko.com/api/v3/coins/markets?vs_currency=inr&per_page=250&order=market_cap_desc&{page}&sparkline=False")

  response = requests.get(url)

  if response.status_code == 200:

     data = response.json()

  records.extend(data)


  print(f"Page: {page} Records: {len(data)}")

In [ ]:
df = pd.DataFrame(records)
df

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df = df[["id", "symbol", "name", "current_price",
         "market_cap", "market_cap_rank","total_volume",
         "circulating_supply", "total_supply","ath", "atl","last_updated"]]
df["last_updated"]=pd.to_datetime(df["last_updated"]).dt.date

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.rename(columns={"last_updated":"date"},inplace=True)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.to_csv("crypto_data.csv", index=False)

Top 3 Coins – Historical Prices

In [ ]:
coin_ids = ["bitcoin", "ethereum", "tether"]

history = []

for coin_id in coin_ids:

    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart?vs_currency=usd&days=365"

    response = requests.get(url)
    data = response.json()

    for timestamp, price in data["prices"]:
        history.append({
            "coin_id": coin_id,
            "date": pd.to_datetime(timestamp, unit="ms"),
            "price_usd": price
        })

history_df = pd.DataFrame(history)

history_df

In [ ]:
history_df.to_csv("top 3 coin_history.csv", index=False)

In [ ]:
history_df.shape

In [ ]:
history_df.columns

In [ ]:
history_df.info()

In [ ]:
history_df.describe()

In [ ]:
history_df["coin_id"].value_counts()

Oil Prices (WTI Crude) bold text

In [ ]:
data = pd.read_csv("https://raw.githubusercontent.com/datasets/oil-prices/main/data/wti-daily.csv")

In [ ]:
oil_df = pd.DataFrame(data)

In [ ]:
oil_df

In [ ]:
oil_df[oil_df["Date"].between("2020-01-01","2026-01-01")]

In [ ]:
oil_df.to_csv("oil_price.csv", index=False)

Stock Prices (Yahoo Finance API)

In [ ]:
pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd

# Choose indices
tickers = ["^GSPC", "^IXIC", "^NSEI"]
start_date = "2020-01-01"
end_date = "2026-08-25"

# Download historical data
stocks_df = yf.download(tickers, start=start_date, end=end_date, group_by="ticker")
stocks_df.head()

In [ ]:
stocks_1 = stocks_df["^NSEI"].reset_index()
stocks_1["Ticker"] = "^NSEI"

stocks_2 = stocks_df["^IXIC"].reset_index()
stocks_2["Ticker"] = "^IXIC"

stocks_3 = stocks_df["^GSPC"].reset_index()
stocks_3["Ticker"] = "^GSPC"

In [ ]:
stocks_final = pd.concat([stocks_1, stocks_2, stocks_3],ignore_index=True)

In [ ]:
stocks_final.columns

In [ ]:
pd.DataFrame(stocks_final)

In [ ]:
stocks_final.shape

In [ ]:
stocks_final.to_csv("stocks_price.csv", index=False)

In [ ]:
pip install mysql-connector

In [ ]:
pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import mysql.connector

load_dotenv()

conn = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

In [ ]:
cursor.execute("use market_analysis")

In [ ]:
cursor.execute("""create table Cryptocurrencies(
    id VARCHAR(100) PRIMARY KEY,
    symbol VARCHAR(10),
    name VARCHAR(100),
    current_price DECIMAL(18,6),
    market_cap BIGINT,
    market_cap_rank INT,
    total_volume BIGINT,
    circulating_supply DECIMAL(30,8),
    total_supply DECIMAL(30,8),
    ath DECIMAL(18,6),
    atl DECIMAL(18,6),
    date DATE)
    """)
conn.commit()

In [ ]:
cursor.execute("""create table crypto_prices(
    coin_id VARCHAR(50),
    date DATE,
    price_usd DECIMAL(18,6),
    FOREIGN KEY (coin_id) REFERENCES Cryptocurrencies(id)
    )
    """)
conn.commit()

In [ ]:
cursor.execute("""create table oil_prices(
date DATE PRIMARY KEY,
    price_usd DECIMAL(18,6)
    )
    """)
conn.commit()

In [ ]:
cursor.execute("""create table Stock_prices(
    date DATE,
    open DECIMAL(18,6),
    high DECIMAL(18,6),
    low DECIMAL(18,6),
    close DECIMAL(18,6),
    volume BIGINT,
    ticker VARCHAR(20)
    )
    """)
conn.commit()

In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv("D:\project1\crypto_data.csv")
df1

In [ ]:
df1 = df1.drop_duplicates(subset=["id"])

In [ ]:
cursor.executemany("""insert into Cryptocurrencies
    (id, symbol, name, current_price, market_cap, market_cap_rank, total_volume, circulating_supply, total_supply, ath, atl, date)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """,df1.values.tolist())
conn.commit()

In [ ]:
df2 = pd.read_csv("D:/project1/top 3 coin_history.csv")
df2

In [ ]:
df2["coin_id"].unique()

In [ ]:
cursor.executemany("""INSERT INTO crypto_prices
    (coin_id, date, price_usd)
    VALUES (%s, %s, %s)
    """,df2.values.tolist())
conn.commit()

In [ ]:
df3 = pd.read_csv("D:\project1\oil_price.csv")
df3

In [ ]:
cursor.executemany("""INSERT INTO oil_prices
    (date, price_usd)
    VALUES (%s, %s)
    """,df3.values.tolist())
conn.commit()

In [ ]:
df4 = pd.read_csv("D:\project1\stocks_price.csv")
df4

In [ ]:
cursor.executemany("""INSERT INTO Stock_prices
    (date, open, high, low, close, volume, ticker)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    """,df4.values.tolist())
conn.commit()

In [ ]:
cursor.execute("""
SELECT id, symbol, name, market_cap, market_cap_rank
FROM cryptocurrencies
ORDER BY market_cap DESC
LIMIT 3
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT id, symbol, name, circulating_supply, total_supply
FROM cryptocurrencies
WHERE total_supply IS NOT NULL
AND circulating_supply > 0.90 * total_supply
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT id, symbol, name, current_price, ath
FROM cryptocurrencies
WHERE current_price >= 0.90 * ath
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT AVG(market_cap_rank) AS average_market_cap_rank
FROM cryptocurrencies
WHERE total_volume > 1000000000
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT id, symbol, name, date
FROM cryptocurrencies
ORDER BY date DESC
LIMIT 1
""")

result = cursor.fetchall()
result

Crypto Prices

In [ ]:
cursor.execute("""
SELECT MAX(price_usd) AS highest_bitcoin_price
FROM crypto_prices
WHERE coin_id = 'bitcoin'
AND date >= CURDATE() - INTERVAL 365 DAY
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT AVG(price_usd) AS average_ethereum_price
FROM crypto_prices
WHERE coin_id = 'ethereum'
AND date >= CURDATE() - INTERVAL 1 YEAR
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT date, price_usd
FROM crypto_prices
WHERE coin_id = 'bitcoin'
AND date BETWEEN '2025-01-01' AND '2025-01-31'
ORDER BY date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT coin_id, AVG(price_usd) AS average_price
FROM crypto_prices
WHERE date >= CURDATE() - INTERVAL 1 YEAR
GROUP BY coin_id
ORDER BY average_price DESC
LIMIT 1
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
(
    (
        AVG(CASE
            WHEN date BETWEEN '2025-09-01' AND '2025-09-30'
            THEN price_usd
        END)
        -
        AVG(CASE
            WHEN date BETWEEN '2024-09-01' AND '2024-09-30'
            THEN price_usd
        END)
    )
    /
    AVG(CASE
        WHEN date BETWEEN '2024-09-01' AND '2024-09-30'
        THEN price_usd
    END)
) * 100 AS percentage_change
FROM crypto_prices
WHERE coin_id = 'bitcoin'
AND date BETWEEN '2024-09-01' AND '2025-09-30'
""")

result = cursor.fetchall()
result

Oil Prices

In [ ]:
cursor.execute("""
SELECT MAX(price_usd) AS highest_oil_price
FROM oil_prices
WHERE date >= CURDATE() - INTERVAL 5 YEAR
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    YEAR(date) AS year,
    AVG(price_usd) AS average_oil_price
FROM oil_prices
GROUP BY YEAR(date)
ORDER BY year
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT date, price_usd
FROM oil_prices
WHERE date BETWEEN '2020-03-01' AND '2020-04-30'
ORDER BY date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT MIN(price_usd) AS lowest_oil_price
FROM oil_prices
WHERE date >= CURDATE() - INTERVAL 10 YEAR
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    YEAR(date) AS year,
    MAX(price_usd) - MIN(price_usd) AS volatility
FROM oil_prices
GROUP BY YEAR(date)
ORDER BY year
""")

result = cursor.fetchall()
result

Stock Prices

In [ ]:
cursor.execute("""
SELECT *
FROM stock_prices
WHERE ticker = '^GSPC'
ORDER BY date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT MAX(close) AS highest_nasdaq_closing_price
FROM stock_prices
WHERE ticker = '^IXIC'
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    date,
    high,
    low,
    (high - low) AS price_difference
FROM stock_prices
WHERE ticker = '^GSPC'
ORDER BY price_difference DESC
LIMIT 5
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    ticker,
    YEAR(date) AS year,
    MONTH(date) AS month,
    AVG(close) AS average_closing_price
FROM stock_prices
GROUP BY ticker, YEAR(date), MONTH(date)
ORDER BY year, month, ticker
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT AVG(volume) AS average_nsei_volume
FROM stock_prices
WHERE ticker = '^NSEI'
AND date BETWEEN '2024-01-01' AND '2024-12-31'
""")

result = cursor.fetchall()
result

Cross-Market JOIN Queries

In [ ]:
cursor.execute("""
SELECT
    AVG(c.price_usd) AS bitcoin_average_price,
    AVG(o.price_usd) AS oil_average_price
FROM crypto_prices c
JOIN oil_prices o
ON c.date = o.date
WHERE c.coin_id = 'bitcoin'
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    (
        COUNT(*) * SUM(c.price_usd * s.close)
        - SUM(c.price_usd) * SUM(s.close)
    )
    /
    SQRT(
        (
            COUNT(*) * SUM(c.price_usd * c.price_usd)
            - POW(SUM(c.price_usd), 2)
        )
        *
        (
            COUNT(*) * SUM(s.close * s.close)
            - POW(SUM(s.close), 2)
        )
    ) AS correlation
FROM crypto_prices c
JOIN stock_prices s
ON c.date = s.date
WHERE c.coin_id = 'bitcoin'
AND s.ticker = '^GSPC'
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    c.date,
    c.price_usd AS ethereum_price,
    s.close AS nasdaq_close
FROM crypto_prices c
JOIN stock_prices s
ON c.date = s.date
WHERE c.coin_id = 'ethereum'
AND s.ticker = '^IXIC'
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY c.date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    o.date,
    o.price_usd AS oil_price,
    c.price_usd AS bitcoin_price,
    (
        (o.price_usd -
        LAG(o.price_usd) OVER (ORDER BY o.date))
        /
        LAG(o.price_usd) OVER (ORDER BY o.date)
    ) * 100 AS oil_change_percent
FROM oil_prices o
JOIN crypto_prices c
ON o.date = c.date
WHERE c.coin_id = 'bitcoin'
ORDER BY o.date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    c.date,
    c.coin_id,
    c.price_usd AS crypto_price,
    s.close AS nifty_close
FROM crypto_prices c
JOIN stock_prices s
    ON c.date = s.date
WHERE s.ticker = '^NSEI'
AND c.coin_id IN ('bitcoin', 'ethereum', 'tether')
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY c.date, c.coin_id
""")

result = cursor.fetchall()
result

In [ ]:
top3_ids = [row[0] for row in top3]

placeholders = ', '.join(['%s'] * len(top3_ids))

query25 = f"""
SELECT
    c.date,
    c.coin_id,
    c.price_usd AS crypto_price,
    s.close AS nifty_close
FROM crypto_prices c
JOIN stock_prices s
    ON c.date = s.date
WHERE s.ticker = '^NSEI'
AND c.coin_id IN ({placeholders})
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY c.date, c.coin_id
"""

cursor.execute(query25, top3_ids)

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    s.date,
    s.close AS sp500_close,
    o.price_usd AS oil_price
FROM stock_prices s
JOIN oil_prices o
ON s.date = o.date
WHERE s.ticker = '^GSPC'
ORDER BY s.date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    (
        COUNT(*) * SUM(c.price_usd * o.price_usd)
        - SUM(c.price_usd) * SUM(o.price_usd)
    )
    /
    SQRT(
        (
            COUNT(*) * SUM(c.price_usd * c.price_usd)
            - POW(SUM(c.price_usd), 2)
        )
        *
        (
            COUNT(*) * SUM(o.price_usd * o.price_usd)
            - POW(SUM(o.price_usd), 2)
        )
    ) AS correlation
FROM crypto_prices c
JOIN oil_prices o
ON c.date = o.date
WHERE c.coin_id = 'bitcoin'
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    c.date,
    c.price_usd AS ethereum_price,
    s.close AS nasdaq_close
FROM crypto_prices c
JOIN stock_prices s
ON c.date = s.date
WHERE c.coin_id = 'ethereum'
AND s.ticker = '^IXIC'
ORDER BY c.date
""")

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT id, symbol, name, market_cap_rank
FROM cryptocurrencies
ORDER BY market_cap_rank
LIMIT 3
""")

top3 = cursor.fetchall()
top3

In [ ]:
top3_ids = [row[0] for row in top3]

placeholders = ', '.join(['%s'] * len(top3_ids))

query29 = f"""
SELECT
    c.date,
    c.coin_id,
    c.price_usd AS crypto_price,
    s.ticker,
    s.close AS stock_close
FROM crypto_prices c
JOIN stock_prices s
    ON c.date = s.date
WHERE c.coin_id IN ({placeholders})
AND s.ticker IN ('^GSPC', '^IXIC', '^NSEI')
AND c.date BETWEEN '2025-01-01' AND '2025-12-31'
ORDER BY c.date, c.coin_id, s.ticker
"""

cursor.execute(query29, top3_ids)

result = cursor.fetchall()
result

In [ ]:
cursor.execute("""
SELECT
    c.date,
    c.price_usd AS bitcoin_price,
    o.price_usd AS oil_price,
    s.close AS sp500_close
FROM crypto_prices c
JOIN oil_prices o
    ON c.date = o.date
JOIN stock_prices s
    ON c.date = s.date
WHERE c.coin_id = 'bitcoin'
AND s.ticker = '^GSPC'
ORDER BY c.date
""")

result = cursor.fetchall()
result